# Reproducing Hazon et al. (2022) from CRCNS hc-30: position decoding

**Paper.** Hazon O, Minces VH, Tomàs DP, Ganguli S, Schnitzer MJ, Jercog PE (2022). Noise
correlations in neural ensemble activity limit the accuracy of hippocampal spatial
representations. *Nature Communications* 13, 4276.
[doi:10.1038/s41467-022-31254-y](https://doi.org/10.1038/s41467-022-31254-y)

**Dataset.** CRCNS hc-30, Jercog P and Schnitzer M (2022), "Functional calcium imaging
recordings of hippocampal area CA1 from mice performing a linear track task",
[doi:10.6080/K0GH9G5X](https://doi.org/10.6080/K0GH9G5X). One file, `hc-30/live/event_traces.mat`
(142,350,100 bytes), streamed from the public S3 bucket `crcnsarchive` into memory.

**Version.** CRCNS data sets are not versioned, so the version is recorded as `unversioned`
(docs/archives.md). The paper cites no version; the only one published is the one in the bucket
(data descriptor version 0.6, July 8, 2022, the initial version), which matches the claims file's
description (110 sessions with mouse name, session ID, sparse event array and position). The
MD5 sum from the data set's `checksums.md5`, which the next cell checks against the streamed
bytes, is `fe0f5dd2fca6d64dbf0cd3f35e31e1f0` (event_traces.mat); the descriptor PDF is
`320d157fbcc43be385485cc186842636`.

**Text the claims were written from.** The version of record (PMC9314334) with its Methods and
Supplementary Information; the paper's code, github.com/approbatory/ML-project at commit
2057af9facfd3035c573e1254bdc32ed4781ba60, was read for method details (claims/hazon-2022.yaml).
Method details taken by this attempt cite the passage or code file in `hc30.py`.

**Data.** 110 sessions of 12 mice with an event-onset array (frames x neurons) and the position
in camera pixels. There are no cell positions, no frame times and no trial table; trials,
bins and the event trace are built here from the Methods and the code (`hc30.py`).

| Claim | Status |
|---|---|
| fig2d-rms-error-ensemble-vs-shuffled | tested |
| fig2cd-shuffle-better-in-all-bins | tested |
| fig2ef-imse-saturates-with-correlations | tested |
| suppl7b-initial-slope-lower-with-correlations | tested |
| saturation-ensemble-size | tested |
| suppl11b-i0-vs-single-cell-snr | tested |
| suppl11c-asymptotic-imse-vs-asymptotic-snr | tested |
| suppl8-task-performance-and-accuracy | tested |
| suppl6-twenty-bins-near-optimal | tested |
| suppl9-fit-recovers-saturation | tested (simulation, data elsewhere) |

The other claims are in `reproduce_hazon_2022.ipynb`. The decoders are the heavy part: this
notebook takes about 40 minutes on 4 CPUs (3 worker processes), with fewer repetitions than the
paper's 80 (stated at each claim).

In [ ]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[_v] = "1"  # one thread per process; the decoders run in 3 processes
import multiprocessing as mp
import resource
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import hc30

T0 = time.time()
sessions, md5, load_s = hc30.load_sessions()
print(f"streamed and parsed event_traces.mat in {load_s:.0f} s; md5 {md5} "
      f"({'matches' if md5 == hc30.MD5 else 'DIFFERS FROM'} checksums.md5)")

# Settings of this notebook (the paper's values in brackets)
REPS_FULL = 10   # repetitions with all neurons [80]
REPS_SUB = 4     # neuron subsets per ensemble size [80]
SIZES = [1] + list(range(25, 500, 25))  # below the session's count, plus all neurons [1, 10, 20, ...]
BIN_COUNTS = [10, 15, 20, 25, 30, 40]  # Supplementary Figure 6 [not stated]
REPS_BINS = 2    # [80]
C_SVM = 1.0      # assumed; see hc30.OvOLinearSVM

## Data check

In [ ]:
rows, TENS = [], []
for i, s in enumerate(sessions):
    T, d, tr = hc30.session_tensor(s)
    TENS.append((T, d))
    perf, n_slow, n_turn = hc30.performance(s["position"], tr)
    rows.append(dict(i=i, mouse=s["mouse"], session=s["session_id"], neurons=T.shape[0],
                     right=int((d == 1).sum()), left=int((d == -1).sum()), performance=perf,
                     slow_errors=n_slow, turn_errors=n_turn))
info = pd.DataFrame(rows)
info["min_trials"] = info[["right", "left"]].min(axis=1)
info["large"] = (info.neurons > 200) & (info.min_trials > 30)
LARGE = info.index[info.large].values
print(f"sessions {len(info)}, mice {info.mouse.nunique()}, neurons {info.neurons.sum()}, "
      f"large-data sessions {len(LARGE)} from {info.loc[LARGE, 'mouse'].nunique()} mice")
print("trials per direction used by the decoder (smaller of the two): median %d, range %d to %d"
      % (info.min_trials.median(), info.min_trials.min(), info.min_trials.max()))
info["clock_err_cm"] = [hc30.clock_check(s, 0) for s in sessions]
MISALIGNED = info.index[info.clock_err_cm > 12].values
print("sessions whose event trace does not track position at lag 0 (quick decoder error > 12 cm,"
      " first notebook):", info.loc[MISALIGNED, ["mouse", "session", "clock_err_cm"]].round(1)
      .values.tolist())
print("performance: median %.3f, range %.3f to %.3f; slow errors median %d, turn-around errors "
      "median %d" % (info.performance.median(), info.performance.min(), info.performance.max(),
                     info.slow_errors.median(), info.turn_errors.median()))

In [ ]:
# Cost estimate before the full run: one repetition on the first session of each mouse.
t = time.time()
probe = info.groupby("mouse").i.first().values
for i in probe:
    T, d = TENS[i]
    hc30.decode_full(T, d, seed=0, reps=1, C=C_SVM)
per_rep = (time.time() - t) / len(probe)
n_sizes = sum(sum(1 for s in SIZES if s < n) for n in info.neurons)
est = per_rep * (len(info) * REPS_FULL + 0.6 * n_sizes * REPS_SUB) / 3 / 60
print(f"{per_rep:.2f} s per repetition (both conditions, all neurons); {len(info) * REPS_FULL} "
      f"full-ensemble repetitions and {n_sizes * REPS_SUB} subset repetitions; a rough lower "
      f"bound of {est:.0f} min on 3 processes (the preparatory run took about three times this)")

## Decoding runs

Every session: REPS_FULL repetitions with all neurons, and REPS_SUB subsets at each ensemble
size below the session's neuron count. Each session has its own seed.

In [ ]:
def job(i):
    T, d = TENS[i]
    full = hc30.decode_full(T, d, seed=20000 + i, reps=REPS_FULL, C=C_SVM)
    sizes = [s for s in SIZES if s < T.shape[0]]
    iu, is_ = hc30.decode_series(T, d, seed=30000 + i, sizes=sizes, reps=REPS_SUB, C=C_SVM)
    full["sizes"] = np.array(sizes + [T.shape[0]])
    full["imse_u"] = np.r_[iu, np.mean(1 / np.array(full["mse_u"]))]
    full["imse_s"] = np.r_[is_, np.mean(1 / np.array(full["mse_s"]))]
    return full

t = time.time()
with mp.get_context("fork").Pool(3) as pool:
    DEC = pool.map(job, range(len(TENS)), chunksize=1)
print(f"decoding: {(time.time() - t) / 60:.1f} min; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")

## fig2d-rms-error-ensemble-vs-shuffled

> The RMS decoding error with all neurons is 9.38 +/- 4.06 cm with noise correlations and
> 5.15 +/- 2.40 cm after trial shuffling (median +/- IQR over 110 sessions), so noise
> correlations impair position estimates.

In [ ]:
info["rms_u"] = [np.mean(np.sqrt(r["mse_u"])) for r in DEC]
info["rms_s"] = [np.mean(np.sqrt(r["mse_s"])) for r in DEC]
for col, name in (("rms_u", "ensemble"), ("rms_s", "shuffled")):
    q1, q3 = np.percentile(info[col], [25, 75])
    print(f"{name}: median {info[col].median():.2f} cm, IQR {q3 - q1:.2f} cm "
          f"(quartiles {q1:.2f} to {q3:.2f}), range {info[col].min():.2f} to {info[col].max():.2f}")
print("ensemble error larger than shuffled in", int((info.rms_u > info.rms_s).sum()), "of",
      len(info), "sessions; Wilcoxon p = %.1e" % stats.wilcoxon(info.rms_u, info.rms_s).pvalue)
ok_clock = ~info.index.isin(MISALIGNED)
for col, name in (("rms_u", "ensemble"), ("rms_s", "shuffled")):
    q1, q3 = np.percentile(info.loc[ok_clock, col], [25, 75])
    print(f"variant without the {len(MISALIGNED)} misaligned sessions, {name}: median "
          f"{info.loc[ok_clock, col].median():.2f} cm, IQR {q3 - q1:.2f} cm")
print("spread over repetitions within a session (SD of RMS), median: ensemble %.2f, shuffled %.2f cm"
      % (np.median([np.std(np.sqrt(r["mse_u"])) for r in DEC]),
         np.median([np.std(np.sqrt(r["mse_s"])) for r in DEC])))
print(info.groupby("mouse")[["rms_u", "rms_s"]].median().round(2))

Variant: the decoder's regularisation. On the first session of each mouse, 2 repetitions with
C = 0.1 and C = 10 against the main C = 1.

In [ ]:
def c_variant(args):
    i, C = args
    T, d = TENS[i]
    r = hc30.decode_full(T, d, seed=40000 + i, reps=2, C=C)
    return i, C, np.mean(np.sqrt(r["mse_u"])), np.mean(np.sqrt(r["mse_s"]))

with mp.get_context("fork").Pool(3) as pool:
    CV = pool.map(c_variant, [(i, C) for i in probe for C in (0.1, 10.0)], chunksize=1)
cv = pd.DataFrame(CV, columns=["i", "C", "rms_u", "rms_s"])
cv = pd.concat([cv, pd.DataFrame({"i": probe, "C": 1.0, "rms_u": info.rms_u[probe].values,
                                  "rms_s": info.rms_s[probe].values})])
print(cv.groupby("C")[["rms_u", "rms_s"]].median().round(2))

**Result: reproduced, resting on the decoder's regularisation.** The median RMS error with all
neurons is 9.36 cm (IQR 3.92) with noise correlations and 4.80 cm (IQR 1.73) after shuffling,
against 9.38 +/- 4.06 and 5.15 +/- 2.40 cm; both medians and the ensemble IQR are within
tolerance, the shuffled IQR is not, and the ensemble error is larger in 109 of 110 sessions.
Without the six sessions whose events do not track position the medians are 9.14 and 4.78 cm.
With C = 0.1 or 10 in place of 1 the medians on 12 sessions change by less than 0.4 cm.

## fig2cd-shuffle-better-in-all-bins

> Decoding is less accurate with noise correlations than without in every spatial bin;
> misclassifications are typically one bin and reach up to three bins.

In [ ]:
def per_bin_rms(conf):
    pos = np.arange(40) // 2
    err2 = (pos[:, None] - pos[None, :]) ** 2 * hc30.BIN_CM ** 2
    by_class = np.sqrt((conf * err2).sum(1) / conf.sum(1))
    by_pos = np.sqrt(np.array([(conf[2 * b:2 * b + 2] * err2[2 * b:2 * b + 2]).sum()
                               / conf[2 * b:2 * b + 2].sum() for b in range(20)]))
    return by_pos, by_class

Bu, Bs, Cu, Cs, accu, accs = [], [], [], [], [], []
for r in DEC:
    a, b = per_bin_rms(r["conf_u"])
    c, e = per_bin_rms(r["conf_s"])
    Bu.append(a); Cu.append(b); Bs.append(c); Cs.append(e)
    accu.append(np.diag(r["conf_u"]) / r["conf_u"].sum(1))
    accs.append(np.diag(r["conf_s"]) / r["conf_s"].sum(1))
Bu, Bs, Cu, Cs = map(np.array, (Bu, Bs, Cu, Cs))
accu, accs = np.array(accu), np.array(accs)
print("(a) 20 positions: session-mean RMS larger for the ensemble in",
      int((Bu.mean(0) > Bs.mean(0)).sum()), "of 20 bins")
print("    ensemble", np.round(Bu.mean(0), 1))
print("    shuffled", np.round(Bs.mean(0), 1))
print("(b) 40 classes: larger for the ensemble in", int((Cu.mean(0) > Cs.mean(0)).sum()), "of 40")
print("correct-class fraction, session average: higher for shuffled in",
      int((accs.mean(0) > accu.mean(0)).sum()), "of 40 classes")
ex = info.index[(info.mouse == "Mouse2022")].values
print("sessions of animal 2022:", list(ex))
for i in ex:
    print(f"  session {i} ({info.session[i]}): correct fraction higher for shuffled in "
          f"{int((accs[i] > accu[i]).sum())} of 40 classes; ties {int((accs[i] == accu[i]).sum())}")
# size of errors
for tag in ("u", "s"):
    conf = sum(r[f"conf_{tag}"] for r in DEC)
    pos = np.arange(40) // 2
    dist = np.abs(pos[:, None] - pos[None, :])
    mis = conf * (dist > 0)
    hist = np.bincount(dist.ravel(), weights=mis.ravel(), minlength=20)
    frac = hist / hist.sum()
    cum = np.cumsum(frac[1:] * 1.0)
    med = int(np.searchsorted(cum, 0.5) + 1)
    largest = int(np.max(np.where(frac >= 0.01)[0]))
    print(f"{'ensemble' if tag == 'u' else 'shuffled'}: misclassified samples {int(hist.sum())}; "
          f"median error {med} bin(s); largest error with >= 1% of them {largest} bins; "
          f"fractions at 1..4 bins {np.round(frac[1:5], 3)}")
fig, ax = plt.subplots(figsize=(5, 3))
x = np.arange(20) + 1
for B, lab in ((Bu, "ensemble"), (Bs, "shuffled")):
    m = B.mean(0)
    ci = 1.96 * B.std(0, ddof=1) / np.sqrt(len(B))
    ax.plot(x, m, label=lab)
    ax.fill_between(x, m - ci, m + ci, alpha=0.3)
ax.set_xlabel("spatial bin"); ax.set_ylabel("RMS error (cm)"); ax.legend()
plt.show()
print("plotted range %.1f to %.1f cm" % (min(Bu.mean(0).min(), Bs.mean(0).min()),
                                       max(Bu.mean(0).max(), Bs.mean(0).max())))

**Result: reproduced.** The session-mean RMS error is larger for the ensemble than for the
shuffled data in 20 of 20 positions and 40 of 40 classes, and the correct-class fraction
averaged over sessions is higher for the shuffled data in 40 of 40 classes (39 to 40 of 40 in
each session of animal 2022). Misclassifications are most often one bin (median 1 bin, 70.6% of
errors); errors of up to 5 bins each hold at least 1% of the misclassified samples with
correlations, and up to 3 bins without.

## fig2ef-imse-saturates-with-correlations

> With noise correlations the decoding accuracy (IMSE) approaches a bound as ensembles grow,
> while without them it keeps increasing; the asymptotic IMSE (I0 N) and the saturation size
> N are larger for the shuffled data.

In [ ]:
fits = []
for i, r in enumerate(DEC):
    I0, N, I0c, Nc = hc30.fit_imse(r["sizes"], r["imse_u"])
    I0s, Ns, I0sc, Nsc = hc30.fit_imse(r["sizes"], r["imse_s"])
    fits.append(dict(I0=I0, N=N, I0N=I0 * N, I0_ci=I0c, N_ci=Nc, I0s=I0s, Ns=Ns, I0Ns=I0s * Ns,
                     I0s_ci=I0sc, Ns_ci=Nsc))
F = pd.DataFrame(fits)
info = pd.concat([info, F], axis=1)
print("fits failed: unshuffled", int(F.N.isna().sum()), "shuffled", int(F.Ns.isna().sum()))
print("negative or absurd fits (N < 0 or N > 1e6): unshuffled", int(((F.N < 0) | (F.N > 1e6)).sum()),
      "shuffled", int(((F.Ns < 0) | (F.Ns > 1e6)).sum()))
print(F[["I0", "N", "I0N", "I0s", "Ns", "I0Ns"]].describe().round(5).T)

def two_tests(df, label):
    a = stats.wilcoxon(df.I0N, df.I0Ns)
    b = stats.wilcoxon(df.N, df.Ns)
    print(f"{label}: I0 N shuffled higher in {int((df.I0Ns > df.I0N).sum())} of {len(df)}, "
          f"p = {a.pvalue:.2e}; N shuffled higher in {int((df.Ns > df.N).sum())} of {len(df)}, "
          f"p = {b.pvalue:.2e}")

ok = F.notna().all(axis=1)
two_tests(info[ok], "110 sessions")
pm = info[ok].groupby("mouse")[["I0N", "I0Ns", "N", "Ns"]].mean()
two_tests(pm, "12 per-animal means")
print("animals with higher mean I0 N for shuffled:", int((pm.I0Ns > pm.I0N).sum()), "of", len(pm))
print(pm.round(4))
# saturation: does unshuffled IMSE at the largest size fall short of the fitted asymptote?
frac_u = [r["imse_u"][-1] / (f["I0N"]) for r, f in zip(DEC, fits)]
frac_s = [r["imse_s"][-1] / (f["I0Ns"]) for r, f in zip(DEC, fits)]
print(f"IMSE at all neurons as a fraction of the fitted asymptote: median unshuffled "
      f"{np.nanmedian(frac_u):.2f}, shuffled {np.nanmedian(frac_s):.2f}")
fig, ax = plt.subplots(figsize=(5, 3))
k = int(info.neurons.idxmax())
ax.plot(DEC[k]["sizes"], DEC[k]["imse_u"], "o-", label="ensemble")
ax.plot(DEC[k]["sizes"], DEC[k]["imse_s"], "o-", label="shuffled")
ax.set_xlabel("neurons"); ax.set_ylabel("IMSE (1/cm^2)"); ax.legend(); ax.set_title(f"session {k}")
plt.show()
print("plotted IMSE range %.4f to %.4f" % (min(DEC[k]["imse_u"].min(), DEC[k]["imse_s"].min()),
                                          max(DEC[k]["imse_u"].max(), DEC[k]["imse_s"].max())))

**Result: reproduced.** Over 110 sessions the fitted asymptotic IMSE (I0 N) is higher for the
shuffled data in 97 sessions (Wilcoxon p = 3.5e-17) and N in 94 (p = 5.2e-15); over the 12 per-
animal means p = 4.9e-04 and 1.5e-03, and the mean I0 N is higher for the shuffled data in 12 of
12 animals. At all neurons the unshuffled IMSE reaches a median 0.49 of its fitted asymptote,
the shuffled IMSE a negligible fraction: the shuffled curves are close to straight lines, and 78
of their fits give N above 1e6, while 7 unshuffled fits are degenerate too.

## suppl7b-initial-slope-lower-with-correlations

> The initial slope I0 of the IMSE curve differs significantly between unshuffled and shuffled
> data across all sessions, which the paper attributes to an artifactual sublinear growth of
> accuracy at small ensemble sizes.

In [ ]:
w = stats.wilcoxon(info.I0[ok], info.I0s[ok])
print(f"I0 median unshuffled {info.I0[ok].median():.5f}, shuffled {info.I0s[ok].median():.5f}; "
      f"shuffled higher in {int((info.I0s[ok] > info.I0[ok]).sum())} of {int(ok.sum())}; "
      f"Wilcoxon p = {w.pvalue:.2e}")

**Result: reproduced.** I0 differs between unshuffled and shuffled data (Wilcoxon p = 9.8e-16
over 110 sessions); it is lower with correlations (median 0.00010 against 0.00019, shuffled
higher in 102 sessions).

## saturation-ensemble-size

> Spatial accuracy approaches saturation (fitted slope 5% of I0) at 346 to 1402 neurons
> across animals, and reaches 95% of the asymptotic IMSE at 1894 to 7671 neurons.

In [ ]:
def gmean(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x) & (x > 0)]
    return np.exp(np.log(x).mean())

for name, sel in (("(a) 73 large-data sessions", info.large & ok), ("(b) 110 working sessions", ok)):
    g = info[sel].groupby("mouse").N.apply(gmean)
    lo, hi = g.min(), g.max()
    print(f"{name}: {len(g)} mice; geometric-mean N {lo:.0f} to {hi:.0f}; 5% slope at "
          f"{lo * (np.sqrt(20) - 1):.0f} to {hi * (np.sqrt(20) - 1):.0f} neurons; 95% of asymptote "
          f"at {19 * lo:.0f} to {19 * hi:.0f}")
    print("   per mouse N:", g.round(0).to_dict())
print("sessions with N <= 0 left out of the geometric means:", int((info.N[ok] <= 0).sum()))
good_fit = ok & (info.N > 0) & (info.N < 1e6) & ~info.index.isin(MISALIGNED)
g = info[good_fit & info.large].groupby("mouse").N.apply(gmean)
print(f"variant, large-data sessions without the misaligned sessions and without fits with N "
      f"outside 0 to 1e6 ({int((good_fit & info.large).sum())} sessions): geometric-mean N "
      f"{g.min():.0f} to {g.max():.0f}; 5% slope at {g.min() * (np.sqrt(20) - 1):.0f} to "
      f"{g.max() * (np.sqrt(20) - 1):.0f}; 19 N {19 * g.min():.0f} to {19 * g.max():.0f}")

**Result: not reproduced.** Under reading (a), 73 large-data sessions, the geometric mean of N
per animal spans 30 to 886, so the 5% slope is reached at 105 to 3,076 neurons and 95% of the
asymptote at 575 to 16,830, against 346 to 1,402 and 1,894 to 7,671; both bounds are outside
tolerance. The lowest animal (2011, N = 30) includes the sessions whose events do not track
position. Reading (b), 110 sessions, gives a far wider range, from degenerate fits in animals
2010 and 2029.

## Asymptotic SNR of the large-data sessions (as in the first notebook)

The same function, seeds and settings as the first notebook (20 subsets per size), so the
values are the same.

In [ ]:
def _sn(i):
    T, d = TENS[i]
    sizes, P, M = hc30.signal_noise_series(T, d, seed=1000 + i, n_subsets=20)
    sig, _ = hc30.slope_with_se(sizes, M[0])
    noi, _ = hc30.slope_with_se(sizes, M[1])
    return sig / noi

t = time.time()
with mp.get_context("fork").Pool(3) as pool:
    asnr = pool.map(_sn, list(LARGE), chunksize=1)
info.loc[LARGE, "asnr"] = asnr
print(f"asymptotic SNR of {len(LARGE)} sessions in {time.time() - t:.0f} s; median {np.median(asnr):.3f}")

## suppl11b-i0-vs-single-cell-snr

> Across mice, the initial slope I0 of the IMSE curve correlates with the average single-cell
> SNR (r squared 0.73, p < 0.01).

In [ ]:
scs = np.array([hc30.single_cell_snr(*TENS[i]) for i in LARGE])
L = info.loc[LARGE].copy()
L["scs_main"], L["scs_mean"], L["scs_median"] = scs[:, 0], scs[:, 1], scs[:, 2]
ag = L.groupby("mouse").mean(numeric_only=True)
for col in ("scs_main", "scs_mean", "scs_median"):
    r, p = stats.pearsonr(ag[col], ag.I0)
    print(f"{col}: across {len(ag)} mice r = {r:.3f}, r^2 = {r * r:.3f}, p = {p:.4f}")
print(ag[["I0", "scs_main", "scs_mean"]].round(5))
Lg = L[good_fit[LARGE].values]
agg_ = Lg.groupby("mouse").mean(numeric_only=True)
r, p = stats.pearsonr(agg_.scs_main, agg_.I0)
print(f"variant without misaligned sessions and fits with N outside 0 to 1e6 ({len(Lg)} sessions):"
      f" r = {r:.3f}, r^2 = {r * r:.3f}, p = {p:.4f}")

**Result: not reproduced.** Across the 10 mice I0 does not correlate with the single-cell SNR
under any reading (r = -0.071, -0.114 and -0.016). The per-mouse mean I0 of animal 2011 is 23.7,
from degenerate fits; the other animals are between 0.00005 and 0.00031.

## suppl11c-asymptotic-imse-vs-asymptotic-snr

> Across mice, the fitted asymptotic IMSE (I0 N) correlates with the asymptotic SNR (r squared
> 0.88).

In [ ]:
Lok = L[np.isfinite(L.I0N)]
ag = Lok.groupby("mouse")[["I0N", "asnr"]].mean()
r, p = stats.pearsonr(ag.I0N, ag.asnr)
print(f"linear: across {len(ag)} mice r = {r:.3f}, r^2 = {r * r:.3f}, p = {p:.2e}")
agl = Lok.assign(lI=np.log10(Lok.I0N.where(Lok.I0N > 0)), la=np.log10(Lok.asnr)).groupby("mouse")[["lI", "la"]].mean()
r, p = stats.pearsonr(agl.lI, agl.la)
print(f"log10:  across {len(agl)} mice r = {r:.3f}, r^2 = {r * r:.3f}, p = {p:.2e}")
r, p = stats.pearsonr(Lok.I0N, Lok.asnr)
print(f"across {len(Lok)} sessions, linear r = {r:.3f}, p = {p:.2e}")
Lg = L[good_fit[LARGE].values]
agg_ = Lg.groupby("mouse")[["I0N", "asnr"]].mean()
r, p = stats.pearsonr(agg_.I0N, agg_.asnr)
print(f"variant, linear, without misaligned sessions and fits with N outside 0 to 1e6 "
      f"({len(Lg)} sessions): r = {r:.3f}, r^2 = {r * r:.3f}, p = {p:.2e}")

**Result: reproduced, on the log scale.** On the log scale the per-mouse I0 N and asymptotic SNR
give r = 0.894, r squared 0.800 and p = 4.8e-04, within tolerance of 0.88 and below 0.001. On
the linear scale r = 0.039, because a few degenerate fits with very large I0 N dominate the per-
mouse means.

## suppl8-task-performance-and-accuracy

> Task performance correlates with the number of trials (r 0.85) and with asymptotic decoding
> accuracy (r 0.52); the number of trials correlates weakly with accuracy (r 0.34), and a path
> analysis shows performance predicts accuracy while trial number does not.

In [ ]:
import statsmodels.api as sm

L["n_trials"] = L.right + L.left
Lp = L[np.isfinite(L.I0N) & (L.I0N > 0)].copy()
Lp["log_I0N"] = np.log10(Lp.I0N)
print(f"{len(Lp)} sessions (of {len(L)}) with a positive I0 N")
r, p = stats.pearsonr(Lp.performance, Lp.n_trials)
print(f"performance vs trials: r = {r:.3f}, p = {p:.1e}")
for acc in ("log_I0N", "I0N"):
    r1, p1 = stats.pearsonr(Lp[acc], Lp.n_trials)
    r2, p2 = stats.pearsonr(Lp[acc], Lp.performance)
    print(f"{acc}: vs trials r = {r1:.3f} (p {p1:.1e}); vs performance r = {r2:.3f} (p {p2:.1e})")
X = sm.add_constant(Lp[["n_trials", "performance"]])
mod = sm.OLS(Lp.log_I0N, X).fit()
print("path model, log10 I0 N ~ trials + performance:")
print(mod.summary2().tables[1][["Coef.", "Std.Err.", "P>|t|"]].round(5))
print("performance range %.3f to %.3f; trials %d to %d" % (Lp.performance.min(), Lp.performance.max(),
                                                           Lp.n_trials.min(), Lp.n_trials.max()))

**Result: not reproduced.** Performance correlates with the number of trials (r = 0.847, p =
3.8e-21), as the paper reports, but the asymptotic accuracy does not correlate with the number
of trials (log10 I0 N, r = -0.054) or with performance (r = 0.189, p = 0.11), against 0.34 and
0.52. In the path model both predictors are significant (performance p = 1.0e-04, trials p =
3.5e-04, with a negative coefficient).

## suppl6-twenty-bins-near-optimal

> Twenty spatial bins give close to the smallest decoding error relative to the continuous
> position (11.75 cm, 95% CI 10.33 to 13.18 cm), similar to 30 bins (11.53 cm).

In [ ]:
def bins_job(args):
    i, nb = args
    try:
        a, b, drop = hc30.decode_bins(sessions[i], nb, seed=50000 + 100 * i + nb, reps=REPS_BINS,
                                      C=C_SVM)
        return i, nb, a.mean(), b.mean(), drop
    except Exception as e:
        print("failed", i, nb, repr(e))
        return i, nb, np.nan, np.nan, np.nan

t = time.time()
with mp.get_context("fork").Pool(3) as pool:
    BR = pool.map(bins_job, [(i, nb) for i in range(len(sessions)) for nb in BIN_COUNTS], chunksize=1)
print(f"bin-count variant: {(time.time() - t) / 60:.1f} min")
br = pd.DataFrame(BR, columns=["i", "bins", "centre", "class_mean", "dropped"])
print("fraction of (trial, bin) samples with no frame, mean over sessions:",
      br.groupby("bins").dropped.mean().round(4).to_dict())
print("failed (session, bins):", br[br.centre.isna()][["i", "bins"]].values.tolist())
summ = br.groupby("bins").agg(n=("centre", "count"), centre=("centre", "mean"),
                              sem_c=("centre", lambda x: x.std(ddof=1) / np.sqrt(x.count())),
                              class_mean=("class_mean", "mean"),
                              sem_m=("class_mean", lambda x: x.std(ddof=1) / np.sqrt(x.count())))
summ["ci_c"] = 1.96 * summ.sem_c
print(summ.round(3))
for col, sem in (("centre", "sem_c"), ("class_mean", "sem_m")):
    m20, s20 = summ.loc[20, col], summ.loc[20, sem]
    below = summ.index[summ[col] < m20 - s20].tolist()
    print(f"{col}: 20 bins {m20:.2f} cm, 30 bins {summ.loc[30, col]:.2f}; bin counts more than one "
          f"SEM below 20: {below}")

**Result: reproduced.** Relative to the continuous position the mean RMS error is 11.84 cm with
20 bins and 11.94 cm with 30 bins, within tolerance of 11.75 and 11.53 cm. 10 and 15 bins give
lower means (11.20 and 11.26 cm), but none is more than one SEM (0.96 cm) below 20 bins. The
prediction at the class mean position gives the same values within 0.06 cm.

## suppl9-fit-recovers-saturation

> On simulated place cells with information-limiting input noise, the IMSE fit gives
> consistent saturation estimates N from ensembles of 500 and 2000 neurons.

A simulation of the model of the Supplementary Figure 9 legend (data: elsewhere); no dataset
values are used. The legend says the decoding follows the paper's methods except for the
Poisson naive Bayes model, so the simulated events are arranged as the paper's tensor: summed
per run and spatial bin, 40 bin and direction classes, two halves of the runs. Choices the
legend leaves open (standard choices named by the criterion): back-and-forth runs at a
constant 30 cm/s over 118 cm, frames at 20 Hz, input noise drawn independently per frame,
peak rate 0.05 events per frame, 20 bins.

In [ ]:
rng9 = np.random.default_rng(909)
L9, FS9, SPEED = hc30.TRACK_CM, 20.0, 30.0
n_frames = int(30 * 60 * FS9)
step = SPEED / FS9
phase = (np.arange(n_frames) * step) % (2 * L9)
true_pos = np.where(phase < L9, phase, 2 * L9 - phase)
direction = np.where(phase < L9, 1, -1)
run_id = (np.arange(n_frames) * step // L9).astype(int)
noisy_pos = true_pos + rng9.normal(0, 10, n_frames)
true_bin = np.clip((true_pos / L9 * 20).astype(int), 0, 19)
cls = 2 * true_bin + (direction == -1)
sigma9 = 30 / (2 * np.sqrt(2 * np.log(2)))
key = run_id * 40 + cls
uk, inv = np.unique(key, return_inverse=True)
s_run, s_cls = uk // 40, uk % 40
s_frames = np.bincount(inv).astype(float)
n_runs = run_id.max() + 1
print(f"{n_runs} runs, {len(uk)} run-bin samples, {s_frames.mean():.1f} frames per sample")

def simulate(n_cells):
    c = rng9.uniform(0, L9, n_cells)
    Y = np.zeros((len(uk), n_cells))
    for a in range(0, n_frames, 6000):
        rate = 0.05 * np.exp(-0.5 * ((noisy_pos[a:a + 6000, None] - c[None, :]) / sigma9) ** 2)
        np.add.at(Y, inv[a:a + 6000], rng9.poisson(rate))
    return Y

def nb_imse(Y, sizes, reps):
    out = []
    for n in sizes:
        v = []
        for _ in range(reps):
            sub = rng9.choice(Y.shape[1], n, replace=False)
            first_runs = rng9.permutation(n_runs)[: n_runs // 2]
            half = np.isin(s_run, first_runs)
            mses = []
            for a, b in ((half, ~half), (~half, half)):
                Ya, Yb = Y[a][:, sub], Y[b][:, sub]
                lam = np.array([Ya[s_cls[a] == k].sum(0) / s_frames[a][s_cls[a] == k].sum()
                                for k in range(40)]) + 1e-6
                ll = Yb @ np.log(lam).T - s_frames[b][:, None] * lam.sum(1)[None, :]
                p = ll.argmax(1)
                mses.append(np.mean(((p // 2 - s_cls[b] // 2) * hc30.BIN_CM) ** 2))
            v.append(1 / np.mean(mses))
        out.append(np.mean(v))
    return np.array(out)

res9 = {}
for n_total, sizes in ((500, [1, 10, 20, 50, 100, 200, 300, 400, 500]),
                       (2000, [1, 10, 20, 50, 100, 200, 300, 400, 500, 1000, 1500, 2000])):
    Y = simulate(n_total)
    im = nb_imse(Y, sizes, reps=16)
    I0, N, I0c, Nc = hc30.fit_imse(sizes, im)
    res9[n_total] = (N, Nc)
    print(f"{n_total} neurons: IMSE {np.round(im, 4)}; fit I0 = {I0:.5f}, N = {N:.0f} +/- {Nc:.0f} (95%)")
(N1, c1), (N2, c2) = res9[500], res9[2000]
print(f"intervals overlap: {(N1 - c1) <= (N2 + c2) and (N2 - c2) <= (N1 + c1)}")

**Result: reproduced.** On the simulated ensembles the fitted N is 704 +/- 554 from 500 neurons
and 338 +/- 87 from 2,000 neurons (95% intervals), and the intervals overlap. The estimate from
500 neurons is imprecise: its interval is six times wider.

## Summary

| Claim | Outcome |
|---|---|
| fig2d-rms-error-ensemble-vs-shuffled | reproduced (assumed: SVM regularisation) |
| fig2cd-shuffle-better-in-all-bins | reproduced |
| fig2ef-imse-saturates-with-correlations | reproduced |
| suppl7b-initial-slope-lower-with-correlations | reproduced |
| saturation-ensemble-size | not reproduced |
| suppl11b-i0-vs-single-cell-snr | not reproduced |
| suppl11c-asymptotic-imse-vs-asymptotic-snr | reproduced |
| suppl8-task-performance-and-accuracy | not reproduced |
| suppl6-twenty-bins-near-optimal | reproduced |
| suppl9-fit-recovers-saturation | reproduced (simulation) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

Differences from the paper's method: the decoder is libsvm's one-vs-one linear SVM with C = 1 and
voting, not MATLAB's fitcecoc with an unregularised SpaRSA fit and loss-weighted decoding; 10
repetitions with all neurons, 4 subsets per ensemble size and sizes in steps of 25, against 80
and steps of 10; 2 repetitions per bin count. The IMSE fits that rest on these fewer, noisier
points are often degenerate, which is the likeliest reason the claims built on I0 and N (the
saturation range, I0 against single-cell SNR, the performance correlations) are not reproduced.

What was found about the dataset: see the first notebook; the sessions whose events do not track
position inflate the decoding error (up to 51.9 cm) and give degenerate fits.

In [ ]:
print(f"total run time {(time.time() - T0) / 60:.1f} min; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")